# Exercise 3 for the class EE-568 Theory and Methods of Reinforcement Learning taught at EPFL in Spring 2026 by Prof. Volkan Cevher

### Names and Sciper numbers

#### Group Member 1:
Name: Rubie Luo

Sciper number: 423142

#### Group Member 2:
Name: Kristina Nordang

Sciper number: 423319

#### Group Member 3:
Name: Oussama Yazidi

Sciper number: 311471

### LLM Usage

You are encouraged **not to use** LLMs or other AI tools so that you can fully engage with and learn the course material. Uploading the full document or copy-pasting the questions into an AI tool are strictly not allowed. 

If you do use them at any point, please clearly and transparently disclose what tool was used, and how in the next cell. 

If AI use is suspected but not clearly explained, we reserve the right to ask follow-up questions to clarify your understanding of the work. 

In [ ]:
%pip install matplotlib torch scipy numpy

### Do the imports -- no need to change this
import numpy as np
from typing import List
import matplotlib.pyplot as plt
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'
import sys
sys.path.insert(0, "src/")
from environment import GridWorldEnvironment
from MDPsolver import MDPsolver
from utils import *
from plot import *
# %load_ext autoreload
# %autoreload 2

***Before starting, we recall the use of the gridworld environment.***

The gridworld environment is instantiated via the class `GridWorldEnvironment`. 

***It takes 4 input values:***
- `reward_mode` : Integer between 0 and 3 for different reward profiles,
- `size`: Gridworld size,
- `prop`: Probability assigned to the event that the agent does not follow the chosen action but another one selected uniformely at random,
- `gamma`: Discount factor of the environment.

***Interface of a Gridworld instance:***
- `print(gridworld.n_states)` # return the number of states
- `print(gridworld.n_actions)` # return the number of actions
- `print(gridworld.r)` # return a matrix where each element indicates the reward corresponding to each (state, action) pair.
- `print(gridworld.gamma)` # return the discount factor
- `print(gridworld.sparseT[action])` # Input: action, Return: a matrix containing the state-to-state transition probabilities for the action passed as input.

<img src="src/vis_gridworld.png" alt="fishy" class="bg-primary" width="400px">

# Ex 1: Prove of the Policy Gradient Theorem via the Performance Difference Lemma (20 points)

Denote $J(\pi) = \langle \mu, V^\pi \rangle$ and recall that the performance difference lemma states
$$
J(\pi) - J(\pi') = \frac{1}{(1-\gamma)}\mathbb{E}_{s \sim \lambda^{\pi'}}[\langle\pi(\cdot|s) - \pi'(\cdot| s) , Q^\pi(s, \cdot) \rangle]
$$
where $\lambda^{\pi'} \in \Delta_{\mathcal{S}\times\mathcal{A}}$ denotes the occupancy measure of the policy $\pi'$.

Now let us consider direct parametization, and compute a partial derivative for the entry of $\pi$ at index $(\bar{s},\bar{a})$:

$$
\frac{\partial J(\pi)}{\partial \pi(\bar{a}|\bar{s})}.
$$

**Questions**

To help you compute this partial derivative, consider the policies $\pi'$ parameterized by some (sufficiently small) $\delta \in \mathbb{R}$ via
$$
    \pi'(a|s) = \begin{cases}
        \pi(\bar{a}|\bar{s}) + \delta \quad (\text{if } (s,a)=(\bar{s},\bar{a}))\\
        \pi(a|s) \quad (\text{else})
    \end{cases}
$$

(1) Argue that
$$
\frac{\partial J(\pi)}{\partial \pi(\bar{a}|\bar{s})} = \frac{1}{(1-\gamma)} \lim_{\delta \rightarrow 0} \frac{\mathbb{E}_{s \sim \lambda^{\pi'}}[\langle\pi(\cdot|s) - \pi'(\cdot| s) , Q^\pi(s, \cdot) \rangle]}{\pi(\bar{a}|\bar{s}) - \pi'(\bar{a}|\bar{s})}.
$$

**Answer**

By definition of the partial derivative, for the smooth perturbation $\pi'$ that differs from $\pi$ only at the coordinate $(\bar s,\bar a)$ (with $\pi'(\bar a|\bar s) = \pi(\bar a|\bar s) + \delta$),

$$
\frac{\partial J(\pi)}{\partial \pi(\bar a|\bar s)}
\;=\; \lim_{\delta\to 0} \frac{J(\pi') - J(\pi)}{\pi'(\bar a|\bar s) - \pi(\bar a|\bar s)}
\;=\; \lim_{\delta\to 0} \frac{J(\pi) - J(\pi')}{\pi(\bar a|\bar s) - \pi'(\bar a|\bar s)}.
$$

The two forms are equal because numerator and denominator both flip sign. Now apply the Performance Difference Lemma to the numerator:

$$
J(\pi) - J(\pi') \;=\; \frac{1}{1-\gamma}\,\mathbb{E}_{s\sim\lambda^{\pi'}}\!\big[\langle \pi(\cdot|s) - \pi'(\cdot|s),\, Q^{\pi}(s,\cdot)\rangle\big].
$$

Substituting:

$$
\boxed{\frac{\partial J(\pi)}{\partial \pi(\bar a|\bar s)}
\;=\; \lim_{\delta\to 0}\, \frac{1}{1-\gamma}\, \frac{\mathbb{E}_{s\sim\lambda^{\pi'}}\!\big[\langle \pi(\cdot|s) - \pi'(\cdot|s),\, Q^{\pi}(s,\cdot)\rangle\big]}{\pi(\bar a|\bar s) - \pi'(\bar a|\bar s)},}
$$

(2) Argue that $$\frac{\partial J(\pi)}{\partial \pi(\bar{a}|\bar{s})} = \lim_{\delta \rightarrow 0} \lambda^{\pi'}(\bar{s}) Q^\pi(\bar{s}, \bar{a}).$$

Hint: Write the expectation in the previous question as a sum and use the fact that $\frac{\pi(a|s) - \pi'(a|s)}{\pi(\bar{a}|\bar{s}) - \pi'(\bar{a}|\bar{s})} = \mathbf{1}_{\{ (\bar{s},\bar{a}) = (s,a) \}}$.

**Answer**

Expand the inner product as a sum over actions and turn the expectation over $s\sim\lambda^{\pi'}$ into an explicit sum over states:

$$
\mathbb{E}_{s\sim\lambda^{\pi'}}\!\big[\langle \pi(\cdot|s) - \pi'(\cdot|s),\, Q^{\pi}(s,\cdot)\rangle\big]
\;=\; \sum_{s} \lambda^{\pi'}(s) \sum_{a} \big(\pi(a|s) - \pi'(a|s)\big)\, Q^{\pi}(s,a).
$$

Divide both sides by $\pi(\bar a|\bar s) - \pi'(\bar a|\bar s)$:

$$
\frac{\mathbb{E}_{s\sim\lambda^{\pi'}}\![\cdot]}{\pi(\bar a|\bar s) - \pi'(\bar a|\bar s)}
\;=\; \sum_{s,a} \lambda^{\pi'}(s)\, \frac{\pi(a|s) - \pi'(a|s)}{\pi(\bar a|\bar s) - \pi'(\bar a|\bar s)}\, Q^{\pi}(s,a).
$$

By construction $\pi$ and $\pi'$ agree on every coordinate except $(\bar s, \bar a)$, so for $(s,a) \ne (\bar s, \bar a)$ the numerator $\pi(a|s) - \pi'(a|s) = 0$  ( NOT SURE ), while at $(s,a) = (\bar s, \bar a)$ the numerator equals the denominator. Using the identity from the hint,

$$
\frac{\pi(a|s) - \pi'(a|s)}{\pi(\bar a|\bar s) - \pi'(\bar a|\bar s)} \;=\; \mathbf{1}_{\{(s,a) = (\bar s, \bar a)\}},
$$

the double sum collapses to a single term:

$$
\sum_{s,a} \lambda^{\pi'}(s)\,\mathbf{1}_{\{(s,a)=(\bar s,\bar a)\}}\, Q^{\pi}(s,a)
\;=\; \lambda^{\pi'}(\bar s)\, Q^{\pi}(\bar s, \bar a).
$$

Combining with part (1):

$$
\boxed{\frac{\partial J(\pi)}{\partial \pi(\bar a|\bar s)} \;=\; \lim_{\delta\to 0}\, \lambda^{\pi'}(\bar s)\, Q^{\pi}(\bar s,\bar a).}
$$

(3) Conclude that $$\frac{\partial J(\pi)}{\partial \pi(\bar{a}|\bar{s})} = \lambda^{\pi}(\bar{s}) Q^\pi(\bar{s}, \bar{a})$$
for the direct parameterization. 

**Answer**

Since $\pi' \to \pi$ as $\delta \to 0$, we get $\lambda^{\pi'}(\bar s) \to \lambda^{\pi}(\bar s)$. Therefore

$$
\boxed{\;\frac{\partial J(\pi)}{\partial \pi(\bar a|\bar s)} \;=\; \lambda^{\pi}(\bar s)\, Q^{\pi}(\bar s,\bar a).\;}
$$

This is exactly the form of the policy gradient under direct (tabular) parameterization.

(4) Prove that for a general parametrization, it holds that
$$
\nabla_\theta J(\pi_\theta) = \sum_{\bar{s},\bar{a}} \lambda^{\pi}(\bar{s}, \bar{a}) Q^\pi(\bar{s}, \bar{a}) \nabla_{\theta} ( \log \pi_\theta(\bar{a}|\bar{s}))
$$

Hint: Use the chain rule to write $$ \nabla_\theta J(\pi_\theta)  = \sum_{\bar{s},\bar{a}} \frac{\partial J(\pi)}{\partial \pi_\theta(\bar{a}|\bar{s})} \nabla_{\theta} \pi_\theta(\bar{a}|\bar{s}), $$
and then use the fact that $\lambda^{\pi}(\bar{s},\bar{a}) = \lambda^{\pi}(\bar{s}) \pi(\bar{a}|\bar{s})$.

**Answer**

For a general parameterization $\pi_\theta$, every entry $\pi_\theta(a|s)$ is a smooth function of $\theta$, so the chain rule gives

$$
\nabla_\theta J(\pi_\theta) \;=\; \sum_{\bar s,\bar a} \frac{\partial J(\pi_\theta)}{\partial \pi_\theta(\bar a|\bar s)}\, \nabla_\theta \pi_\theta(\bar a|\bar s).
$$

Substitute the expression from part (3):

$$
\nabla_\theta J(\pi_\theta) \;=\; \sum_{\bar s,\bar a} \lambda^{\pi}(\bar s)\, Q^{\pi}(\bar s,\bar a)\, \nabla_\theta \pi_\theta(\bar a|\bar s).
$$

Use the log-derivative trick $\nabla_\theta \pi_\theta(\bar a|\bar s) = \pi_\theta(\bar a|\bar s)\, \nabla_\theta \log \pi_\theta(\bar a|\bar s)$ to rewrite the gradient and pull the policy weight inside:

$$
\nabla_\theta J(\pi_\theta) \;=\; \sum_{\bar s,\bar a} \lambda^{\pi}(\bar s)\, \pi_\theta(\bar a|\bar s)\, Q^{\pi}(\bar s,\bar a)\, \nabla_\theta \log \pi_\theta(\bar a|\bar s).
$$

Finally, by definition the state-action occupancy factors as $\lambda^{\pi}(\bar s,\bar a) = \lambda^{\pi}(\bar s)\, \pi(\bar a|\bar s)$, giving the desired form:

$$
\boxed{\;\nabla_\theta J(\pi_\theta) \;=\; \sum_{\bar s,\bar a} \lambda^{\pi}(\bar s,\bar a)\, Q^{\pi}(\bar s,\bar a)\, \nabla_\theta \log \pi_\theta(\bar a|\bar s).\;}
$$



# Ex 2: Natural Policy Gradient with softmax parameterization (20 points)

Recall that the iterates $\{\pi^t\}^{\infty}_{t=1}$ produced by NPG read as follows:
$$
\pi^{t+1}(a|s) = \frac{\pi^t(a|s)e^{\eta Q^{\pi^t}(s,a) }}{\sum_{a'} \pi^t(a'|s) e^{\eta Q^{\pi^t}(s,a')}}.
$$

Implement NPG for an arbitrary step size $\eta$. Please note that $e^{\eta Q^{\pi^t}(s,a)}$ can be zero, account for that in your implementation.

***Hint:*** When computing the exponential update, think about numerical stability. You can subtract the same constant from all values (e.g., the maximum) before applying exp.

In [ ]:
def evaluate_policy(pi, env, tol=1e-10):
    """Implementation of policy evaluation through iteratively applying using a certain policy 
    Args:
        pi: a policy stochastic passed with shape n_states times n_actions
        env: environment
        tol: a scalar to dermerminate whether the policy evaluation convergences
    Returns:
        v: an array with the values of the actions chosen
        q: an array with the q values    
    """
    v = np.zeros(env.n_states)
    q = np.zeros((env.n_states, env.n_actions))
    while True:
        v_old = np.copy(v)
        for a in range(env.n_actions):
            q[:, a] = env.r[:, a] + env.gamma * env.sparseT[a].dot(v)
        for s in range(env.n_states):
            v[s] = q[s].dot(pi[s])
        if np.linalg.norm(v - v_old) < tol:
            break
    return v, q

def npg_update(q, eta, old_policy):
    """Implementation of a greedy approach to choose policies (policy improvement)
    Args:
        q: q values obtained from evaluating the policies
    Returns:
        new_policy: the updates policy
    """
    policy = np.zeros_like(q)
    for s in range(q.shape[0]):
        policy[s] = old_policy[s] * np.exp(eta * q[s] - np.max(eta * q[s])) # ??? (unnormalized update)
        total = np.sum(policy[s])
        if total == 0:
             policy[s] = np.ones(q.shape[1])/q.shape[1]
        else:
            policy[s] = policy[s] / total # ??? normalize
    return policy

def get_greedy_policy(q):
    """Implementation of a greedy approach to choose policies (policy improvement)
    Args:
        q: q values obtained from evaluating the policies
    Returns:
        policy: greedy policy (list)
    """
    policy = np.zeros_like(q)
    for s in range(q.shape[0]):
        policy[s,np.argmax(q[s,:])] = 1
    return policy

In [ ]:
def NPG(env, eta): # apply NPG iterations for 30 steps
    vs = []
    policies = []
    v = np.zeros(env.n_states)
    q = np.zeros((env.n_states, env.n_actions))
    pi = np.ones_like(q)/env.n_actions
    for k in range(30):
        v_old = np.copy(v)
        v, q = evaluate_policy(pi, env)
        if eta < np.inf:
            pi = npg_update(q, eta, pi)
        else:
            pi = get_greedy_policy(q)
        vs.append(v)
        policies.append(pi)
    return vs, policies

Now, we run NPG for different stepsizes in the usual gridworld environment

In [ ]:
reward_mode = 2
size = 10 
prop = 0
gamma=0.99
gridworld = GridWorldEnvironment(reward_mode, size, prop=0, gamma=gamma)
mu = np.ones(gridworld.n_states)/gridworld.n_states
etas = [1e-3, 1e-2, 1e-1, 1, 100, 1e7, np.inf]
v_different_etas = []
pi_different_etas = []
for eta in etas:
    values_pi, policies = NPG(gridworld, eta=eta)
    v_different_etas.append(values_pi)
    pi_different_etas.append(policies)

In [ ]:
solver = MDPsolver(gridworld)
solver.value_iteration()

In [ ]:
# if this plot appears with a too large legend, rerun this line once more
plot_log_lines([np.array([mu.dot(solver.v - v) for v in v_different_etas[i]]) for i, _ in enumerate(etas)], [f"Subopt for eta {eta}" for eta in etas], ["Iteration", "Subopt"], "figs", "NPG.pdf", show = True)

**Question**

Show that NPG with $\eta = \infty$ coincides with Policy Iteration (PI).

More formally: Assuming that $a^\star_s := \mathrm{argmax}_a Q^{\pi^t}(s,a)$ is unique for all $s$, prove that $$ \lim_{\eta \rightarrow \infty} \frac{\pi^t(a|s)e^{\eta Q^{\pi^t}(s,a) }}{\sum_{a'} \pi^t(a'|s) e^{\eta Q^{\pi^t}(s,a')}} = \begin{cases} 1 \quad \text{if} \quad a = a^\star_s \\ 0 \quad \text{otherwise} \end{cases},$$
and explain how this relates to PI.

**Answer**

Let $a^\star_s = \arg\max_a Q^{\pi^t}(s,a)$ (assumed unique). To take the limit cleanly, factor $e^{\eta Q^{\pi^t}(s, a^\star_s)}$ out of both numerator and denominator. For any action $a$,

$$
\frac{\pi^t(a|s)\, e^{\eta Q^{\pi^t}(s,a)}}{\sum_{a'} \pi^t(a'|s)\, e^{\eta Q^{\pi^t}(s,a')}}
\;=\; \frac{\pi^t(a|s)\, e^{\eta\,(Q^{\pi^t}(s,a) - Q^{\pi^t}(s,a^\star_s))}}{\sum_{a'} \pi^t(a'|s)\, e^{\eta\,(Q^{\pi^t}(s,a') - Q^{\pi^t}(s,a^\star_s))}}.
$$

Now examine the exponents $\Delta(a) := Q^{\pi^t}(s,a) - Q^{\pi^t}(s,a^\star_s) \le 0$, with equality iff $a = a^\star_s$. Therefore as $\eta \to \infty$:

- If $a = a^\star_s$: $\Delta(a) = 0$ so $e^{\eta \Delta(a)} = 1$ for all $\eta$.
- If $a \ne a^\star_s$: $\Delta(a) < 0$ (strict, by uniqueness of the argmax), so $e^{\eta \Delta(a)} \to 0$.

In the denominator, only the $a' = a^\star_s$ term survives in the limit, leaving $\pi^t(a^\star_s|s)$ (which is positive under softmax initialization and remains positive under NPG updates). Consequently:

$$
\lim_{\eta\to\infty} \frac{\pi^t(a|s)\, e^{\eta Q^{\pi^t}(s,a)}}{\sum_{a'} \pi^t(a'|s)\, e^{\eta Q^{\pi^t}(s,a')}}
\;=\; \begin{cases} \dfrac{\pi^t(a^\star_s|s)}{\pi^t(a^\star_s|s)} = 1, & a = a^\star_s,\\[2pt] \dfrac{0}{\pi^t(a^\star_s|s)} = 0, & a \ne a^\star_s. \end{cases}
$$

**Connection to PI.** Policy Iteration alternates between (i) policy evaluation, computing $Q^{\pi^t}$, and (ii) greedy improvement, $\pi^{t+1}(a|s) = \mathbf{1}\{a = \arg\max_a Q^{\pi^t}(s,a)\}$. The display above shows the $\eta = \infty$ NPG update produces exactly that greedy policy: it concentrates all probability mass on the unique optimal action under $Q^{\pi^t}$. So NPG with $\eta = \infty$ is mechanically identical to PI on the tabular softmax simplex — the same evaluation step (computing $Q^{\pi^t}$) followed by the same improvement step (going greedy).

**Question**

Is this observation in line with the empirical results in the plot above? I.e., is the plot for $\eta = \infty$ as you would expect it for PI?

**Answer**

Yes, the plot is consistent with what we expect from PI. 

From Lecture 4 Slide 55, under softmax parameterization the NPG update is:

$$\pi_{t+1}(a|s) = \frac{\pi_t(a|s) \exp(\eta Q^{\pi_t}(s,a))}{\sum_{a'} \pi_t(a'|s) \exp(\eta Q^{\pi_t}(s,a'))}$$

As shown in the assignment, as eta approaches infinity this converges to the greedy policy (putting all mass on the argmax action), which is exactly the PI update.

In the plot, eta = infinity would show the fastest convergence since each step takes the maximally aggressive greedy update, just like PI converges rapidly in the tabular setting.

# Ex 2.2 Slow Changing Property of NPG

In this exercise you will investigate by how much consecutive iterates $\pi^t$ and $\pi^{t+1}$ produced by NPG differ and how this distance is controlled by the step size $\eta$.

Plot $$\max_{s \in \mathcal{S}} || \pi^{t+1}(a|s) - \pi^t(a|s) ||_1$$ for different values of $\eta$.

In [ ]:
def compute_policy_variation(policies):
    variation = []
    for pi, pip in zip(policies[1:], policies[:-1]):
        variation.append(np.max([ np.sum(np.abs(pi[s]-pip[s])) for s in range(pi.shape[0])])) # ???
    return variation

In [ ]:
plot_lines(np.array([ compute_policy_variation(np.array(pi_different_etas)[i])
                           for i, _ in enumerate(etas)]), 
               [f" eta = {eta}" for eta in etas], 
               ["Iteration", "Variation"], "figs", "NPG.pdf", show = True)

**Question**

Empirically, is the largest change (among all iterations) between consecutive iterations is larger for smaller or large values of $\eta$?

Empirically, larger eta produces larger per-step policy changes.

This is consistent with the theoretical bound derived in Ex 2.2:

$$\frac{1}{2}\|\pi_{t+1}(s) - \pi_t(s)\|_1^2 \leq \frac{\eta^2}{2(1-\gamma)^2}$$

The right-hand side grows with eta, thus larger step sizes lead to bigger changes between consecutive policies.

This can be seen in the plot, for small eta like 0.001 and 0.01, the variation starts low and decreases smoothly, while for large eta (100 to inf) the variation starts near the maximum value of 2.0 in early iterations before dropping sharply once the policy converges.

Intuitively, when eta is small, the exponential weights exp(eta * Q(s,a)) are all close to 1, so the update barely moves the policy. When eta is large, the highest Q action gets exponentially more weight, causing a large shift in the policy distribution.

## Some Theory to Motivate the Observation Above

**Question**

Our goal is to prove that $$ || \pi^{t+1}(\cdot|s) - \pi^t(\cdot|s) ||_1 \leq \frac{\eta}{1 - \gamma} \quad \forall s \in \mathcal{S}, \forall t \in [T].$$

We guide you towards this result by breaking the proof into small steps.

1) Prove that $$ \frac{1}{2} || \pi^{t+1}(s) - \pi^t(s) ||^2_1 \leq \mathbb{E}_{a \sim \pi^{t+1}(\cdot|s)}[\eta Q^{\pi^t}(s,a)] - \log \bigg(\sum_{a'\in\mathcal{A}} \pi^t(a'|s) \exp (\eta Q^{\pi^t}(s,a'))\bigg) $$

Hint: First apply Pinkser's inequality https://en.wikipedia.org/wiki/Pinsker%27s_inequality to prove that $$\frac{1}{2} || \pi^{t+1}(s) - \pi^t(s) ||^2_1 \leq KL(\pi^{t+1}(s)||\pi^t(s)), $$ then plug in the formula for $\pi^{t+1}$ into the KL term.

**Answer**

Pinsker's inequality applied to the two distributions $\pi^{t+1}(\cdot|s)$ and $\pi^{t}(\cdot|s)$ on the action set gives

$$
\frac{1}{2}\, \|\pi^{t+1}(\cdot|s) - \pi^{t}(\cdot|s)\|_1^2 \;\le\; \mathrm{KL}\!\left(\pi^{t+1}(\cdot|s)\,\|\,\pi^{t}(\cdot|s)\right).
$$

Now compute the KL divergence using the closed form of the NPG update. Let $Z_t(s) := \sum_{a'} \pi^t(a'|s)\, e^{\eta Q^{\pi^t}(s,a')}$ denote the normalizer. Then $\pi^{t+1}(a|s) = \pi^t(a|s)\, e^{\eta Q^{\pi^t}(s,a)} / Z_t(s)$, so

$$
\log\frac{\pi^{t+1}(a|s)}{\pi^{t}(a|s)} \;=\; \eta\, Q^{\pi^t}(s,a) - \log Z_t(s).
$$

Therefore

$$
\mathrm{KL}\!\left(\pi^{t+1}\,\|\,\pi^{t}\right)
= \sum_a \pi^{t+1}(a|s)\,\big[\eta\, Q^{\pi^t}(s,a) - \log Z_t(s)\big]
= \mathbb{E}_{a\sim \pi^{t+1}(\cdot|s)}\!\big[\eta Q^{\pi^t}(s,a)\big] - \log Z_t(s),
$$

since $\sum_a \pi^{t+1}(a|s) = 1$. Combining with Pinsker:

$$
\boxed{\;\frac{1}{2}\,\|\pi^{t+1}(s) - \pi^{t}(s)\|_1^2 \;\le\; \mathbb{E}_{a\sim \pi^{t+1}(\cdot|s)}\!\big[\eta Q^{\pi^t}(s,a)\big] \;-\; \log\!\bigg(\sum_{a'}\pi^t(a'|s)\, e^{\eta Q^{\pi^t}(s,a')}\bigg).\;}
$$

2) Prove that 
$$
\sum_{a\in \mathcal{A}} \pi^{t+1}(a|s) \exp(- \eta Q^{\pi^t}(s,a)) = \frac{1}{\sum_{a'\in \mathcal{A}} \pi^t(a|s) \exp(\eta Q^{\pi^t}(s,a) )}.
$$

**Answer**

Plug the NPG closed form $\pi^{t+1}(a|s) = \pi^t(a|s)\, e^{\eta Q^{\pi^t}(s,a)} / Z_t(s)$ directly into the left-hand side:

$$
\sum_{a} \pi^{t+1}(a|s)\, e^{-\eta Q^{\pi^t}(s,a)}
= \sum_{a} \frac{\pi^t(a|s)\, e^{\eta Q^{\pi^t}(s,a)}}{Z_t(s)}\, e^{-\eta Q^{\pi^t}(s,a)}
= \frac{1}{Z_t(s)} \sum_{a} \pi^t(a|s)
= \frac{1}{Z_t(s)}.
$$

The exponentials cancel pairwise inside the sum, and what remains is just $\sum_a \pi^t(a|s) = 1$. Therefore

$$
\boxed{\;\sum_{a\in\mathcal{A}} \pi^{t+1}(a|s)\, e^{-\eta Q^{\pi^t}(s,a)} \;=\; \frac{1}{\sum_{a'\in\mathcal{A}} \pi^{t}(a'|s)\, e^{\eta Q^{\pi^t}(s,a')}}.\;}
$$

3) Using the results in 1) and 2) prove that 

$$ \frac{1}{2} || \pi^{t+1}(s) - \pi^t(s) ||^2_1 \leq \mathbb{E}_{a \sim \pi^{t+1}(\cdot|s)}[\eta Q^{\pi^t}(s,a)] + \log \bigg(\sum_{a'\in\mathcal{A}} \pi^{t+1}(a'|s) \exp (-\eta Q^{\pi^t}(s,a'))\bigg). $$

**Answer**

From step 2, taking the log of both sides:

$$
\log\!\bigg(\sum_a \pi^{t+1}(a|s)\, e^{-\eta Q^{\pi^t}(s,a)}\bigg)
= -\log\!\bigg(\sum_{a'} \pi^t(a'|s)\, e^{\eta Q^{\pi^t}(s,a')}\bigg),
$$

so we can substitute the negative of the log-normalizer in step 1 with the log-sum on the LHS just above. The bound from step 1 then reads

$$
\frac{1}{2}\,\|\pi^{t+1}(s) - \pi^t(s)\|_1^2
\;\le\; \mathbb{E}_{a\sim\pi^{t+1}}[\eta Q^{\pi^t}(s,a)] \;-\; \log\!\bigg(\sum_{a'}\pi^t(a'|s)\, e^{\eta Q^{\pi^t}(s,a')}\bigg)
$$

$$
=\; \mathbb{E}_{a\sim\pi^{t+1}}[\eta Q^{\pi^t}(s,a)] \;+\; \log\!\bigg(\sum_{a'}\pi^{t+1}(a'|s)\, e^{-\eta Q^{\pi^t}(s,a')}\bigg).
$$

This is the desired inequality. The advantage of this form is that the log term is now an expectation under $\pi^{t+1}$ — exactly the right setup for applying Hoeffding's lemma in step 4.

4) Using Hoeffding's Lemma https://en.wikipedia.org/wiki/Hoeffding%27s_lemma (on the sum in the log term!) and the fact that $$-\frac{1}{1-\gamma} \leq Q^{\pi^t}(s,a) \leq \frac{1}{1-\gamma},$$ conclude that 
$$\frac{1}{2} || \pi^{t+1}(s) - \pi^t(s) ||^2_1 \leq \frac{\eta^2}{2 (1 - \gamma)^2}.$$

**Answer**

Hoeffding's lemma states: if $X$ is a random variable bounded almost surely in $[a,b]$, then for every $\lambda \in \mathbb{R}$,

$$
\log \mathbb{E}\!\left[e^{\lambda(X-\mathbb{E}[X])}\right] \;\le\; \frac{\lambda^2 (b-a)^2}{8}, \quad\text{equivalently}\quad \log \mathbb{E}[e^{\lambda X}] \;\le\; \lambda\, \mathbb{E}[X] + \frac{\lambda^2(b-a)^2}{8}.
$$

Apply it to $X = -\eta\, Q^{\pi^t}(s,a)$ with $a \sim \pi^{t+1}(\cdot|s)$ and $\lambda = 1$. Since $|Q^{\pi^t}(s,a)| \le \frac{1}{1-\gamma}$, the random variable $X$ is bounded between $-\frac{\eta}{1-\gamma}$ and $+\frac{\eta}{1-\gamma}$, so $b - a = \frac{2\eta}{1-\gamma}$. Then

$$
\log\!\bigg(\sum_{a'}\pi^{t+1}(a'|s)\, e^{-\eta Q^{\pi^t}(s,a')}\bigg)
= \log \mathbb{E}_{a\sim \pi^{t+1}}\!\big[e^{-\eta Q^{\pi^t}(s,a)}\big]
\;\le\; -\,\mathbb{E}_{a\sim \pi^{t+1}}[\eta Q^{\pi^t}(s,a)] + \frac{(2\eta/(1-\gamma))^2}{8}.
$$

The last term simplifies to $\frac{\eta^2}{2(1-\gamma)^2}$. Plug this into the bound from step 3:

$$
\frac{1}{2}\,\|\pi^{t+1}(s) - \pi^{t}(s)\|_1^2
\;\le\; \mathbb{E}_{a\sim\pi^{t+1}}[\eta Q^{\pi^t}(s,a)] \;+\; \Big(\!-\mathbb{E}_{a\sim\pi^{t+1}}[\eta Q^{\pi^t}(s,a)] + \frac{\eta^2}{2(1-\gamma)^2}\Big) \;=\; \frac{\eta^2}{2(1-\gamma)^2}.
$$

The two expectation terms cancel exactly, and we obtain

$$
\boxed{\;\frac{1}{2}\,\|\pi^{t+1}(s) - \pi^{t}(s)\|_1^2 \;\le\; \frac{\eta^2}{2(1-\gamma)^2}, \qquad \text{equivalently}\qquad \|\pi^{t+1}(s) - \pi^{t}(s)\|_1 \;\le\; \frac{\eta}{1-\gamma}.\;}
$$

This is the slow-changing property: the per-iterate $\ell_1$ change is controlled linearly by the step size $\eta$, which is the key technical tool for proving NPG convergence.

# Ex 3: OPPO: The importance of Exploration in Policy Gradient (20 points)

In this exercise, we will investigate how crucial it is to perform exploration. That is, adding bonuses to avoid suffering the mismatch coefficients in the convergence bounds.

Let us recall that the standard sample based version of NPG suffers the mismatch coeffcients in the bounds (see Slide 22 in Lecture 5). Those are avoided by OPPO ( See slide 30 in Lecture 5 ).

**To see clearly the advatange of OPPO we will consider an MDP with unbounded mismatch coefficients**

**Question: example of unbounded mismatch coefficients**

Consider a 10 x 10 gridworld, the initial state is always the bottom right corner, i.e. the initial distribution $\mu$ equals $1$ at this starting state and it is zero everywhere else. Can you compute a finite bound for 
$$\max_\pi \max_{s \in \mathcal{S}} \bigg |\frac{\lambda^\pi(s)}{\mu(s)} \bigg|,$$
i.e. the mismatch coefficient? If not, argue for which reason.

**Answer**



No, we cannot compute a finite bound since $\mu(s) = 0$ for most states and the ratio $\frac{\lambda_\mu^{\pi}(s)}{\mu(s)}$ would be undefined for any state $s$ with $\mu(s) = 0$ and $\lambda_\mu^{\pi}(s) > 0$.

[TODO just check this idk if this explanation is enough]

In the following, we experiment with OPPO with and without bonuses in this environment.

***Hint:*** When computing the exponential update, think about numerical stability. You can subtract the same constant from all values (e.g., the maximum) before applying exp.

In [ ]:
reward_mode = 0
size = 10
gamma=0.999
gridworld = GridWorldEnvironment(reward_mode, size, prop=0, gamma=gamma)
r_max = np.max(gridworld.r)
r_min = np.min(gridworld.r)
gridworld.r = (gridworld.r - r_min) / (r_max - r_min)

In [ ]:
from copy import deepcopy
np.random.seed(0)
def oppo(K: int = 10000, H: int = 20, beta: float = 0.0001, eta=5) -> List[float]:
    """
    Function implementing OPPO with UCB bonuses algorithm.

    :param K: Number of episodes, positive int
    :param H: Number of steps per episode, positive int
    :param beta: Algorithm hyperparameter, constant which scales the bonuses, positive float

    :return: reward after each step, list of K * H floats
    """

    # Initialize tabular records
    rewards = []
    Q = H * np.ones((H, gridworld.n_states, gridworld.n_actions))
    V = H * np.ones((H + 1, gridworld.n_states))
    policy = H * np.ones((H, gridworld.n_states, gridworld.n_actions))/gridworld.n_actions
    V[H, :] = 0
    estimated_transitions = np.ones((H, gridworld.n_states, 
                                     gridworld.n_actions, 
                                     gridworld.n_states))/gridworld.n_states
    N = np.zeros((H, gridworld.n_states, gridworld.n_actions))
    bonus = np.zeros((H, gridworld.n_states, gridworld.n_actions))
    N_next = np.zeros((H, gridworld.n_states, gridworld.n_actions, gridworld.n_states))

    for k in range(K):  # Episode loop
        state = 99  # Initial state
        for h in range(H):
            #NPG Update
            policy[h, state, :] = policy[h, state, :] * np.exp(eta * (Q[h, state, :] - np.max(Q[h, state, :]))) # ??? # (unnormalized)
            
            total = np.sum(policy[h, state, :])
            if total == 0 or np.any(np.isnan(policy[h, state, :])):
                policy[h, state, :] = np.ones(gridworld.n_actions) / gridworld.n_actions # ??? # fallback to a uniform distribution
            else:
                policy[h, state, :] = policy[h, state, :] / np.sum(policy[h, state, :])# ??? # normalize

            # Sample one action the current policy
            a = np.random.choice(gridworld.n_actions, p=policy[h, state, :]) # ???
            rewards.append(gridworld.r[state, a])

            # Record that we visited this state-action pair (again)
            N[h, state, a] += 1 # ??? 

            # Get the new state according to the transition dynamics
            new_state = np.random.choice(gridworld.n_states,
                                         p=gridworld.T[a][state])
            N_next[h, state, a, new_state] += 1 # ???
            
            # Update empirical transition estimate
            estimated_transitions[h,state,a,:] = N_next[h, state, a, :] / (N[h, state, a] + 1) # ???
            
            state = new_state
                    
        # Calculate the UCB bonus
        bonus = beta / np.sqrt(N + 1)

        Q_new = np.zeros_like(Q)
        V_new = np.zeros_like(V)
        for h in reversed(range(H)):  # Step loop
            # Update Q according to the algorithm
            Q_new[h] = np.clip(gridworld.r + bonus[h] + estimated_transitions[h].dot(V_new[h + 1]), 0, H - h + 1) # ???

            # Update V as the Q-value of the optimal actions for the current state
            for state in range(gridworld.n_states):
                V_new[h, state] = policy[h, state, :].dot(Q_new[h, state, :]) # ??? #
        Q = deepcopy(Q_new)
        V = deepcopy(V_new)
    return rewards

In [ ]:
to_plot = []
betas = [0, 1e-5, 1e-3, 0.1, 10]

for beta in betas:
    print(beta)
    reward_OPPO = oppo(beta = beta)  # You can play around with the arguments if you like
    to_plot.append(np.cumsum(reward_OPPO))

In [ ]:
labels = [ f"OPPO beta = {beta}" for beta in betas]
plot_lines(
    to_plot,
    labels,
    ["Iteration", "Reward collected so far"],
    "figs",
    "ucbvseps",
    show=False
)

**Question**

Why does setting $\beta = 0$ lead to bad results? 

*Hint: Explain using the remarks in slide 28 and the theoretical bound in Slide 22 of Lecture 5*.

**Answer**


Setting $\beta = 0$ means no exploration bonus is added. From Slide 22, the convergence bound for sample-based NPG includes the term $\sqrt{\kappa \epsilon_{\text{stat}}}$, where

$$\kappa = \left\| \frac{\lambda_\mu^{\pi^\star}}{\mu} \right\|_\infty$$

In this environment, $\mu$ is 0 for all states except the bottom right corner, making $\kappa = \infty$ for those states. The agent starts in the same corner every episode and has no incentive to visit other states. This is exactly the problem OPPO is designed to fix. The bonus $\text{bonus}_h(s,a) \propto 1/\sqrt{N_h(s,a)}$ encourages visiting less explored state (Slide 27).

**Question**

Why does setting $\beta$ too large lead to poor results?

*Hint: Answer using the regret bound for OPPO given at the beginning of slide 30.*

**Answer**

When $\beta$ is too large, the bonuses $\text{bonus}_h^t(s,a) = \beta / \sqrt{N_h^t(s,a)}$ 
dominate the reward signal $$Q_h^t(s,a) = r_h(s,a) + \text{bonus}_h^t(s,a) + \sum_{s'}\hat{P}_h(s'|s,a)V_{h+1}^t(s')$$
The Q values become driven almost entirely by bonuses and the policy optimizes for visiting new states insread of collecting reward. The cumulative reward grows slowly because the agent explores and ignores the actual reward structure of the environment. 

[TODO CHECK CUZ IDK HOW THE REGRET BOUND IN SLIDE 30 IS RELEVANT? IDK I DIDNT USE IT TO EXPLAIN THIS IDK IF THATS A PROBLEM]

# Ex 4: REINFORCE with parametrized policies (20 points)

In this exercise, we will investigate the effect of choosing different baselines in the reinforce implementation.
This topic is covered from Slide 31 on in Lecture 5.

**Hint: You may want to use Google Colab to run the experiments faster, but you don't have to.**

### Import the Necessary Packages

In [ ]:
# TODO: you may need to run this to make sure to have the correct versions
!pip install gym==0.25.2
!pip install gym-notices==0.0.8

In [ ]:
import gym
import numpy as np
np.bool8 = np.bool_ # added to stop error

from collections import deque
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (16, 10)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical
torch.manual_seed(0)

import base64, io

# For visualization
from gym.wrappers.monitoring import video_recorder
from IPython.display import HTML
from IPython import display
import glob

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

### Instantiate the Environment and Agent

The CartPole environment is very simple. It has discrete action space (2) and 4 dimensional state space.

In [ ]:
env = gym.make('CartPole-v0')
env.seed(0)

In [ ]:
class Policy(nn.Module): # definie the policy network
    def __init__(self, state_size=4, action_size=2, hidden_size=32):
        super(Policy, self).__init__()
        self.fc1 = nn.Linear(state_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, action_size)

    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = self.fc2(x)
        return F.softmax(x, dim=1) # we just consider 1 dimensional probability of action

    def act(self, state):
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        probs = self.forward(state).cpu()
        model = Categorical(probs)
        action = model.sample()
        return action.item(), model.log_prob(action)
    

In [ ]:
# REINFORCE (with reward-to-go)
# --> with gradient estimator according to version 2 of the PG theorem (not using Q-values, but reward to go)
def reinforce_rwd2go(policy, optimizer, early_stop=False, n_episodes=1000, max_t=1000, gamma=1.0, print_every=100):
    scores_deque = deque(maxlen=100)
    scores = []
    for e in range(1, n_episodes):
        saved_log_probs = []
        rewards = []
        state = env.reset()
        # Collect trajectory
        for t in range(max_t):
            # Sample the action from current policy
            action, log_prob = policy.act(state)
            saved_log_probs.append(log_prob)
            state, reward, done, _ = env.step(action)
            rewards.append(reward)
            if done:
                break
        # Calculate total expected reward
        scores_deque.append(sum(rewards))
        scores.append(sum(rewards))

        # Recalculate the total reward applying discounted factor
        discounts = [gamma ** i for i in range(len(rewards) + 1)]
        rewards_to_go = [sum([discounts[j]*rewards[j+t] for j in range(len(rewards)-t) ]) for t in range(len(rewards))]

        # Calculate the loss
        policy_loss = []
        for i in range(len(saved_log_probs)):
            log_prob = saved_log_probs[i]
            G = rewards_to_go[i]
            # Note that we are using Gradient Ascent, not Descent. So we need to calculate it with negative rewards.
            policy_loss.append(-log_prob * G)
        # After that, we concatenate whole policy loss in 0th dimension
        policy_loss = torch.cat(policy_loss).sum()

        # Backpropagation
        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

        if e % print_every == 0:
            print('Episode {}\tAverage Score: {:.2f}'.format(e, np.mean(scores_deque)))
        if early_stop and np.mean(scores_deque) >= 195.0:
            print('Environment solved in {:d} episodes!\tAverage Score: {:.2f}'.format(e - 100, np.mean(scores_deque)))
            break
    return scores

**Question**

1. Find **two** good baselines that improve over the implementation of REINFORCE without baseline. You should plot their results below.

You can take inspiration from the Example Notebook we attached for lecture 4, but you **cannot use exactly the same**.

2. Explain why you chose your baselines and why you think they are reasonable.

*Note:* You may also change other parameters such as the learning rate, as long as you clearly state it in your response.

**Answer**

Baseline 1 penalizes large pole angles linearly:

$$b_1(s) = \max(0,\ 50 \cdot (0.2 - |\theta|))$$

So the more tilted the pole, the less reward we expect.

Baseline 2 additionally penalizes angular velocity:

$$b_2(s) = \max(0,\ 50 \cdot (0.2 - |\theta|) - 0.5 \cdot |\dot{\theta}|)$$

Even if the angle is small, a fast angular velocity means the pole is about to fall, 
so expected future reward should be lower.

These baselines were chosen intuitively based on knowledge of the cartpole system. If the goal is to balance and keep the pole upright, it makes sense to penalize a larger angle and additionally a larger velocity. They're reasonable because the plot shows both baselines learn faster and more stably than no baseline. Baseline 2 does better because it captures more information about the state. 

In [ ]:
def naive_baseline(state): # Example Baseline from lecture 4 (for inspiration)
  angle = state[2]
  value = 100*(0.25-angle**2) # TO BE CHANGED USING YOUR BASELINE
  return value

def baseline_1(state): # TO BE CHANGED USING YOUR BASELINE 1
  angle = state[2]
  # linear penalty on angle, scaled to match typical returns
  return max(0, 50 * (0.2 - abs(angle)))

def baseline_2(state): # TO BE CHANGED USING YOUR BASELINE 2
  angle = state[2]
  ang_vel = state[3]
  return max(0, 50 * (0.2 - abs(angle)) - 0.5 * abs(ang_vel))

# PLOT 3: reward-to-go with baseline REINFORCE
# --> with gradient estimator according to version 3 of the PG theorem (not using Q-values, but reward to go)
# --> here, we consider only fixed (handcrafted) baseline functions b : S -> R; clearly, training a NN to predict V^{\pi}(s) as a baseline is also possible (and interesting!)
def reinforce_rwd2go_baseline(policy, optimizer, early_stop=False, baseline=naive_baseline, n_episodes=1000, max_t=1000, gamma=1.0, print_every=100):
    scores_deque = deque(maxlen=100)
    scores = []
    for e in range(1, n_episodes):
        saved_log_probs = []
        rewards = []
        baseline_values = []
        state = env.reset()
        # Collect trajectory
        for t in range(max_t):
            # Sample the action from current policy
            action, log_prob = policy.act(state)
            saved_log_probs.append(log_prob)
            state, reward, done, _ = env.step(action)
            rewards.append(reward)
            baseline_values.append(baseline(state))
            if done:
                break
        # Calculate total expected reward
        scores_deque.append(sum(rewards))
        scores.append(sum(rewards))

        # Recalculate the total reward applying discounted factor
        discounts = [gamma ** i for i in range(len(rewards) + 1)]
        rewards_to_go = [sum([discounts[j]*rewards[j+t] for j in range(len(rewards)-t) ]) for t in range(len(rewards))]

        # Calculate the loss
        policy_loss = []
        for i in range(len(saved_log_probs)):
            log_prob = saved_log_probs[i]
            G_centered = rewards_to_go[i] - baseline_values[i]
            # Note that we are using Gradient Ascent, not Descent. So we need to calculate it with negative rewards.
            policy_loss.append(-log_prob * G_centered)
        # After that, we concatenate whole policy loss in 0th dimension
        policy_loss = torch.cat(policy_loss).sum()

        # Backpropagation
        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

        if e % print_every == 0:
            print('Episode {}\tAverage Score: {:.2f}'.format(e, np.mean(scores_deque)))
        if early_stop and np.mean(scores_deque) >= 195.0:
            print('Environment solved in {:d} episodes!\tAverage Score: {:.2f}'.format(e - 100, np.mean(scores_deque)))
            break
    return scores

In [ ]:
env = gym.make('CartPole-v0')
env.seed(0)

# PLOT 1: run REINFORCE
policy_rwd2go = Policy().to(device)
optimizer_rwd2go = optim.Adam(policy_rwd2go.parameters(), lr=1e-2)
scores_rwd2go = reinforce_rwd2go(policy_rwd2go, optimizer_rwd2go, early_stop=False, n_episodes=2000)

env = gym.make('CartPole-v0')
env.seed(0)

# PLOT 2: run REINFORCE and YOUR baseline 1
policy_baseline_1 = Policy().to(device)
optimizer_baseline_1 = optim.Adam(policy_baseline_1.parameters(), lr=1e-2)
scores_baseline_1 = reinforce_rwd2go_baseline(policy_baseline_1, optimizer_baseline_1, baseline=baseline_1, early_stop=False, n_episodes=2000)

env = gym.make('CartPole-v0')
env.seed(0)

# PLOT 3: run REINFORCE and YOUR baseline 2
policy_baseline_2 = Policy().to(device)
optimizer_baseline_2 = optim.Adam(policy_baseline_2.parameters(), lr=1e-2)
scores_baseline_2 = reinforce_rwd2go_baseline(policy_baseline_2, optimizer_baseline_2, baseline=baseline_2, early_stop=False, n_episodes=2000)



In [ ]:
### Plot the learning progress

# Create the plot
fig = plt.figure(figsize=(20, 6))
ax = fig.add_subplot(111)

# Plot the scores with specified colors and labels
ax.plot(np.arange(1, len(scores_rwd2go) + 1), scores_rwd2go, color='green', label='No Baseline')
ax.plot(np.arange(1, len(scores_baseline_1) + 1), scores_baseline_1, color='blue', label='Baseline 1')
ax.plot(np.arange(1, len(scores_baseline_2) + 1), scores_baseline_2, color='red', label='Baseline 2')

# Set the labels with a larger font size
ax.set_ylabel('Total reward (= time balanced)', fontsize=20)
ax.set_xlabel('Episode #', fontsize=20)

# Set the tick labels to a larger font size
ax.tick_params(axis='both', which='major', labelsize=15)

# Add a legend with a specified font size
ax.legend(fontsize=20)

# Show the plot
plt.show()

# $Q^\star$: Policy Gradient with continuous actions and bound on the bonuses count in OPPO (20 points)
***Question 1:*** Consider using a Gaussian parameterized policy $\pi_{\mu,\Sigma}$ with mean $\mu \in \mathrm{R}^d$ 
and covariance matrix $\Sigma$ . Write down the following gradients:

$$ \nabla_\mu J(\pi_{\mu, \Sigma}) = ???TODO$$
$$ \nabla_\Sigma J(\pi_{\mu, \Sigma}) = ???TODO$$


## Answer

For a **scalar action** $a\in\mathbb{R}$, the Gaussian policy seen in Lecture 4 (slide 7) is
$$
\pi_\theta(a\mid s)\;=\;\frac{1}{\sqrt{2\pi}\,\sigma_\theta(s)}\,\exp\!\left(-\frac{(a-\mu_\theta(s))^2}{2\,\sigma_\theta(s)^2}\right).
$$

For a **$d$-dimensional action** $a\in\mathbb{R}^d$ with mean $\mu\in\mathbb{R}^d$ and covariance matrix $\Sigma\in\mathbb{R}^{d\times d}$ (symmetric positive-definite), the multivariate Gaussian density is
$$
\pi_{\mu,\Sigma}(a)\;=\;\frac{1}{(2\pi)^{d/2}\,|\Sigma|^{1/2}}\,\exp\!\left(-\tfrac{1}{2}(a-\mu)^{\top}\Sigma^{-1}(a-\mu)\right).
$$

Taking the log,
$$
\log\pi_{\mu,\Sigma}(a)\;=\;-\tfrac{d}{2}\log(2\pi)\;-\;\tfrac{1}{2}\log|\Sigma|\;-\;\tfrac{1}{2}(a-\mu)^{\top}\Sigma^{-1}(a-\mu).
$$

We will compute the score functions $\nabla_\mu \log\pi$ and $\nabla_\Sigma \log\pi$, then plug them into the trajectory-level REINFORCE expression of the policy gradient theorem:
$$
\nabla_\theta J(\pi_\theta)\;=\;\mathbb{E}_{\tau\sim p_\theta}\!\left[R(\tau)\sum_{t=0}^{\infty}\nabla_\theta\log\pi_\theta(a_t\mid s_t)\right].
$$

### Gradient with respect to $\mu$

Only the quadratic term in $\log\pi_{\mu,\Sigma}(a)$ depends on $\mu$. Using
$$
\nabla_\mu\!\left[(a-\mu)^{\top}\Sigma^{-1}(a-\mu)\right]\;=\;-2\,\Sigma^{-1}(a-\mu),
$$

$$
\boxed{\;\nabla_\mu \log\pi_{\mu,\Sigma}(a)\;=\;\Sigma^{-1}(a-\mu)\;}
$$

Plugging into REINFORCE,
$$
\boxed{\;\nabla_\mu J(\pi_{\mu,\Sigma})\;=\;\mathbb{E}_{\tau\sim p_{\mu,\Sigma}}\!\left[R(\tau)\sum_{t=0}^{\infty}\Sigma^{-1}(a_t-\mu)\right]\;}
$$

### Gradient with respect to $\Sigma$

Two standard matrix-calculus identities (e.g. *Matrix Cookbook*):
$$
\nabla_\Sigma \log|\Sigma|\;=\;\Sigma^{-1},
\qquad
\nabla_\Sigma\,(a-\mu)^{\top}\Sigma^{-1}(a-\mu)\;=\;-\,\Sigma^{-1}(a-\mu)(a-\mu)^{\top}\Sigma^{-1}.
$$
$$
\nabla_\Sigma \log\pi_{\mu,\Sigma}(a)\;=\;-\tfrac{1}{2}\Sigma^{-1}\;+\;\tfrac{1}{2}\Sigma^{-1}(a-\mu)(a-\mu)^{\top}\Sigma^{-1},
$$
which we can factor as
$$
\boxed{\;\nabla_\Sigma \log\pi_{\mu,\Sigma}(a)\;=\;\tfrac{1}{2}\,\Sigma^{-1}\big[(a-\mu)(a-\mu)^{\top}-\Sigma\big]\Sigma^{-1}\;}
$$

Plugging into REINFORCE,
$$
\boxed{\;\nabla_\Sigma J(\pi_{\mu,\Sigma})\;=\;\mathbb{E}_{\tau\sim p_{\mu,\Sigma}}\!\left[R(\tau)\sum_{t=0}^{\infty}\tfrac{1}{2}\,\Sigma^{-1}\big[(a_t-\mu)(a_t-\mu)^{\top}-\Sigma\big]\Sigma^{-1}\right]\;}
$$

***Question 2*** In this exercise, you will bound the state action counts. This is a crucial part of the OPPO convergence proof. Let $N^t_h(s,a)$ denotes the number of times the state action pair $s,a$ has been visited at step $h$ in all the episode up to $t$ included. Moreover,
let $s^t_h,a^t_h$ be the state action pair visited at step $h$ of the $t^{th}$ episode. Then, prove that 
$$ \sum^T_{t=1} \sum^H_{h=1} \frac{1}{N^t_h(s^t_h, a^t_h)+1} \leq SAH \log( T H)$$

## Answer to Question 2

**Proof.** Reorder the double sum by grouping the summands according to which triple $(s,a,h)$ is being visited. By definition of $N^t_h$, at the $j$-th visit to $(s,a)$ at step $h$ the counter equals $j$, so that visit contributes $\frac{1}{j+1}$. Each triple is visited at most $T$ times (one visit per episode at most), hence:

$$
\sum_{t=1}^{T}\sum_{h=1}^{H}\frac{1}{N^t_h(s^t_h,a^t_h)+1}
\;\le\;
\sum_{h=1}^{H}\sum_{(s,a)}\sum_{j=1}^{T}\frac{1}{j+1}.
$$

$$
\sum_{j=1}^{T}\frac{1}{j+1}\;\le\;\int_{0}^{T}\frac{dx}{x+1}\;=\;\log(T+1)\;\le\;\log(TH).
$$
and : 
$$
\sum_{h=1}^{H}\sum_{(s,a)}\sum_{j=1}^{T}\frac{1}{j+1} = SAH \sum_{j=1}^{T}\frac{1}{j+1}

$$
$$
\boxed{\implies \sum_{t=1}^{T}\sum_{h=1}^{H}\frac{1}{N^t_h(s^t_h,a^t_h)+1}\;\le\;SAH\,\log(TH).\qquad \blacksquare }
$$

***Question 3*** Use the fact above to prove the following bound at slide 30 of Lecture 5. That is, for $\mathrm{bonus}^t_h(s,a) = \frac{H}{\sqrt{N^t_h(s,a)+1}}$ it holds that
$$ \sum^T_{t=1} \sum^H_{h=1} \mathrm{bonus}(s^t_h, a^t_h) \leq H^2\sqrt{ SA T \log( T H)}$$

## Answer to Question 3

**Proof.** Apply Cauchy–Schwarz to the $TH$ summands:
$$
\sum_{t=1}^{T}\sum_{h=1}^{H}\frac{1}{\sqrt{N^t_h(s^t_h,a^t_h)+1}}
\;\le\;\sqrt{TH\,\cdot\,\sum_{t=1}^{T}\sum_{h=1}^{H}\frac{1}{N^t_h(s^t_h,a^t_h)+1}}.
$$

By Question 2:
$$
\sum_{t,h}\frac{1}{N^t_h(s^t_h,a^t_h)+1}\;\le\;SAH\log(TH).
$$

Combining,
$$
\sum_{t,h}\frac{1}{\sqrt{N^t_h(s^t_h,a^t_h)+1}}\;\le\;\sqrt{TH\cdot SAH\log(TH)}\;=\;\sqrt{H^{2}\,SAT\log(TH)}\;=\;H\sqrt{SAT\log(TH)}.
$$

Multiplying by the factor $H$ in $\mathrm{bonus}^t_h$:
$$
\boxed{\sum_{t=1}^{T}\sum_{h=1}^{H}\mathrm{bonus}^t_h(s^t_h,a^t_h)
\;=\;H\!\sum_{t,h}\frac{1}{\sqrt{N^t_h(s^t_h,a^t_h)+1}}
\;\le\;H^{2}\sqrt{SAT\log(TH)}\;\blacksquare}
$$